# FedLabel on OrganAMNIST: reproducing the ICCV 2023 paper

Paper protocol (ICCV 2023 + arXiv appendix):
- 10 clients, Dirichlet alpha, 30% participation (3 clients per round), ResNet18
- 5% labeled (Table 3 / 4) or 20% labeled (Fig. 4b), labeled/unlabeled split uniformly at random per client
- pooled data split 80% train / 5% val / 15% test (`--split paper`), 3 partition seeds, mean (std) reported
- RandAugment(1, 10), variance confidence, lambda_0 = 1.0
- beta: the appendix ablation (Fig. 8) is best around 0.3 to 0.5, so 0.5 is used here

FedAvg / FedAvg (100%) baselines run in the same runner, on the same data and scored on the same global test set.
Run the cells top to bottom; each experiment cell can be run on its own after the setup cells.

In [ ]:
# 1) Drive, repo (branch test1), requirements
import os
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)

REPO_URL = 'https://github.com/ozanyildirim22/federated-learning.git'
REPO_DIR = '/content/federated-learning'
BRANCH   = 'test1'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git fetch -q origin
!git checkout -q {BRANCH} 2>/dev/null || git checkout -q -b {BRANCH} origin/{BRANCH}
!git reset -q --hard origin/{BRANCH}
!git log -1 --oneline
!pip install -q fedlab medmnist path

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (Runtime > Change runtime type > GPU)')

In [ ]:
# 2) Helpers: data cache on Drive, experiment runner, result table
import os, re, glob, shutil, statistics
from IPython import get_ipython

DRIVE_ROOT = '/content/drive/MyDrive/federated-learning-data'
LOG_DIR    = os.path.join(DRIVE_ROOT, 'logs', 'organamnist_paper')
os.makedirs(LOG_DIR, exist_ok=True)

COMMON = dict(rounds=800, clients_per_round=3, batch_size=20, lr=0.01,
              beta=0.5, lambda_0=1.0, tau=15, tau_prime=15)


def prepare_data(alpha, label_ratio, seed):
    """data/organamnist/pickles_paper_alpha{a}_label{l}_seed{s}: local -> Drive zip -> preprocess."""
    name       = f'pickles_paper_alpha{alpha}_label{label_ratio}_seed{seed}'
    local_base = os.path.join(REPO_DIR, 'data', 'organamnist')
    local_dir  = os.path.join(local_base, name)
    drive_dir  = os.path.join(DRIVE_ROOT, 'organamnist')
    drive_zip  = os.path.join(drive_dir, f'{name}.zip')
    if os.path.isdir(local_dir):
        print(f'[local] {name}')
    elif os.path.isfile(drive_zip):
        shutil.unpack_archive(drive_zip, local_base)
        print(f'[drive] extracted {name}')
    else:
        get_ipython().system(
            f'cd {REPO_DIR} && PYTHONPATH=. python data/organamnist/preprocess.py '
            f'--split paper --alpha {alpha} --label_ratio {label_ratio} --seed {seed}')
        if not os.path.isdir(local_dir):
            raise RuntimeError(f'{local_dir} was not created, check the output above')
        os.makedirs(drive_dir, exist_ok=True)
        tmp = shutil.make_archive(f'/content/_{name}', 'zip', root_dir=local_base, base_dir=name)
        shutil.copy(tmp, drive_zip); os.remove(tmp)
        print(f'[drive] cached {name}.zip')


def run(algorithm, alpha, label_ratio, seed, **over):
    """algorithm: fedlabel | fedavg | fedavg_full. The log goes to Drive."""
    p = {**COMMON, **over}
    prepare_data(alpha, label_ratio, seed)
    tag = f'{algorithm}_a{alpha}_l{label_ratio}_b{p["beta"]}_s{seed}'
    get_ipython().system(
        f'cd {REPO_DIR} && PYTHONPATH=. python fedlabel/main.py '
        f'--algorithm {algorithm} --dataset organamnist --model resnet18 '
        f'--split paper --seed {seed} --alpha {alpha} --label_ratio {label_ratio} '
        f'--comms_round {p["rounds"]} --client_num_per_round {p["clients_per_round"]} '
        f'--batch_size {p["batch_size"]} --local_lr {p["lr"]} '
        f'--beta {p["beta"]} --lambda_0 {p["lambda_0"]} --tau {p["tau"]} --tau_prime {p["tau_prime"]} '
        f'--confidence_metric variance --global_test True --eval_every 50 '
        f'2>&1 | tee {LOG_DIR}/{tag}.txt')


# Paper numbers (OrganAMNIST column): (alpha, label_ratio, algorithm) -> (mean, std)
PAPER = {
    (0.1, 0.05, 'fedlabel'):    (69.43, 1.58),   # Table 3
    (0.1, 0.05, 'fedavg'):      (41.90, 2.18),
    (0.1, 0.05, 'fedavg_full'): (71.09, 1.00),
    (1.0, 0.05, 'fedlabel'):    (79.76, 0.83),   # Table 4
    (1.0, 0.05, 'fedavg'):      (59.74, 1.52),
    (1.0, 0.05, 'fedavg_full'): (81.49, 0.96),
}


def summary():
    ansi = re.compile(r'\x1b\[[0-9;]*m')
    rows = {}
    for f in sorted(glob.glob(os.path.join(LOG_DIR, '*.txt'))):
        m = re.match(r'(fedlabel|fedavg_full|fedavg)_a([\d.]+)_l([\d.]+)_b([\d.]+)_s(\d+)\.txt', os.path.basename(f))
        if not m:
            continue
        text = ansi.sub('', open(f, errors='ignore').read())
        acc = re.findall(r'Global .+ accuracy: ([\d.]+)%', text)
        if not acc:
            continue  # run not finished
        alg, a, l, b, s = m.groups()
        rows.setdefault((float(a), float(l), alg, float(b)), []).append(float(acc[-1]))
    print(f'{"alpha":>5} {"label":>5} {"algorithm":<12} {"beta":>4} {"seeds":>5} {"ours":>14} {"paper":>14}')
    for (a, l, alg, b), accs in sorted(rows.items()):
        sd = statistics.pstdev(accs) if len(accs) > 1 else 0.0
        ours = f'{statistics.mean(accs):.2f} ({sd:.2f})'
        pm = PAPER.get((a, l, alg))
        paper = f'{pm[0]:.2f} ({pm[1]:.2f})' if pm else '-'
        print(f'{a:>5} {l:>5} {alg:<12} {b:>4} {len(accs):>5} {ours:>14} {paper:>14}')

## Table 3: alpha = 0.1, 5% labeled (main result)
Paper: FedLabel 69.43 (1.58), FedAvg 41.90 (2.18), FedAvg (100%) 71.09 (1.00).
Start with seed 0 of all three, then seeds 1 and 2.

In [ ]:
run('fedlabel', alpha=0.1, label_ratio=0.05, seed=0)

In [ ]:
run('fedavg', alpha=0.1, label_ratio=0.05, seed=0)

In [ ]:
run('fedavg_full', alpha=0.1, label_ratio=0.05, seed=0)

In [ ]:
for seed in (1, 2):
    run('fedlabel', alpha=0.1, label_ratio=0.05, seed=seed)
    run('fedavg', alpha=0.1, label_ratio=0.05, seed=seed)
    run('fedavg_full', alpha=0.1, label_ratio=0.05, seed=seed)

## Table 4: alpha = 1.0, 5% labeled (low heterogeneity)
Paper: FedLabel 79.76 (0.83), FedAvg 59.74 (1.52), FedAvg (100%) 81.49 (0.96).

In [ ]:
for seed in (0, 1, 2):
    run('fedlabel', alpha=1.0, label_ratio=0.05, seed=seed)
    run('fedavg', alpha=1.0, label_ratio=0.05, seed=seed)
    run('fedavg_full', alpha=1.0, label_ratio=0.05, seed=seed)

## Fig. 4b: alpha = 0.1, 20% labeled
The paper shows this one only as a curve (no number in a table); FedLabel beats the baselines by 2 to 10 points.

In [ ]:
for seed in (0, 1, 2):
    run('fedlabel', alpha=0.1, label_ratio=0.2, seed=seed)
    run('fedavg', alpha=0.1, label_ratio=0.2, seed=seed)

## Optional: beta check
The paper does not state the beta used in its tables; this shows how much it matters here.

In [ ]:
run('fedlabel', alpha=0.1, label_ratio=0.05, seed=0, beta=0.95)

## Results vs paper (final-round accuracy on the pooled 15% test set, mean (std) over seeds)

In [ ]:
summary()